<a href="https://colab.research.google.com/github/linhb03/Ai118Project/blob/dev/Refund_Policy0.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install langchain-openai
!pip install langgraph
!pip install llama_index

  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.79-py3-none-any.whl (449 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.0.0
    Uninstalling langchain-core-1.0.0:
      Successfully uninstalled langchain-core-1.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.0.1 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.79 which is incompatible.
  Using cached langchain_core-1.0.0-py3-none-any.whl.metadata (3.4 kB)
Using cached langchain_core-1.0.0-py3-none-any.whl (467 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
ERROR: pip's dependency resolver does not currently take into account 

In [36]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

# Initialize the model
# It will automatically use the API key you set in the previous step
model = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    openai_api_key=userdata.get('OPENAI_API_KEY')
)

print("ChatOpenAI model is ready.")

ChatOpenAI model is ready.


In [37]:
from langchain_core.tools import tool

@tool
def get_refund_policy_status(days_since_purchase: int) -> str:
    """
    Checks if an order is eligible for a refund based on the number of days since purchase.
    It returns the company's policy and the eligibility status.
    The standard return window is 30 days.
    """

    policy_days = 30

    if days_since_purchase <= policy_days:
        return (
            f"The order is ELIGIBLE for a refund. "
            f"Our policy allows for a full refund within {policy_days} days of the purchase date."
        )
    else:
        return (
            f"The order is NOT ELIGIBLE for a refund. "
            f"Our policy only allows returns within {policy_days} days, and that window has passed."
        )

print("Tool 'get_refund_policy_status' has been created.")

Tool 'get_refund_policy_status' has been created.


In [42]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage, HumanMessage
import uuid
from langgraph.checkpoint.memory import MemorySaver


# The system prompt defines the agent's role and instructions
refunds_system_prompt = SystemMessage(
    """You are a professional Customer Support agent specializing in refunds.
    Your role is to answer customer questions about the refund policy.
    To answer these questions, you will ONLY use the available tools.
    You MUST extract the number of days from the user's query to use the tool.
    Do not make up information.
    """
)

refunds_tools = [get_refund_policy_status]

refund_agent = create_react_agent(
    model=model,
    tools=refunds_tools
)


# --- 2. RUN A TEST CONVERSATION ---

# Set up memory to maintain the conversation
checkpointer = MemorySaver()
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

def run_refund_chatbot(input_text):
    """A function to send a message to the agent and print the response."""
    response = refund_agent.invoke({"messages": [refunds_system_prompt, HumanMessage(content=input_text)]}, config=config)
    print(f"USER: {input_text}")
    print(f"AGENT: {response['messages'][-1].content}\n" + "-"*40)

# Start the test
print("Starting conversation with the Refund Policy Agent (OpenAI)....\n")
run_refund_chatbot("Hello, I want to return the product")
run_refund_chatbot("May I know how long I can return the product?")
run_refund_chatbot("What about an order I received 45 days ago?")

/tmp/ipython-input-285902829.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  refund_agent = create_react_agent(


Starting conversation with the Refund Policy Agent (OpenAI)....

USER: Hello, I want to return the product
AGENT: To assist you with the return process, could you please let me know how many days it has been since you purchased the product? This will help me determine if your order is eligible for a refund based on our policy.
----------------------------------------
USER: May I know how long I can return the product?
AGENT: The standard return window for a product is 30 days from the date of purchase. If you have any specific questions about your purchase, please let me know!
----------------------------------------
USER: What about an order I received 45 days ago?
AGENT: The order you received 45 days ago is not eligible for a refund. Our policy allows returns only within 30 days, and that window has already passed.
----------------------------------------
